In [1]:
import sys
sys.path.append("..")
# from models import iresnet
from collections import OrderedDict
# from termcolor import cprint
from torch.nn import Parameter
import torch.nn.functional as F
import torch.backends.cudnn as cudnn
import numpy as np
import math
import torch
import torch.nn as nn

![](images/magface.png)

N: batch_size

fi: feature vector hay embedding. Trong code là x_norm/x và reseach sử dụng embedding 512-D

wi: vector tâm class identity

theta yi: góc (fi, wi)

s: hệ số tỷ lệ để phóng đại giá trị cos(theta yi...) do nó chỉ nằm trong khoảng [-1,1] rất nhỏ nên phải phóng đại nó ra. Là 
hyper parameter, **recommend s=64**

ai: độ dài feature vector fi. ai=||fi||

LMag: loss tích lũy được trên mỗi batch, trong đó Li là loss của từng sample.

m(ai): angular margin adaptive. **Là 1 hàm tuyến tính**

λg: hyperparameter. Nếu nó càng cao, mô hình tập chung và g(ai), tức kiến trúc struture intra-class chuẩn hơn. Càng nhỏ thì ưu 
tiên phần softmax phục vụ cho classification identity.

[la, ua]: Biên của ai là lower-bound (giới hạn dưới) và upper-bound (giới hạn trên). MagFace kiểm soát độ lớn của embedding. la 
giúp đảm bảo các embedding có mức độ biểu diễn tốt hơn và ua ngăng embedding có giá trị quá lớn, giúp giảm độ phức tạp của mô 
hình và giảm over fitting. **Trong thực nghiệm reseach chọn la=10 và ua=110**

g(ai): Phải là hàm lồi, đơn điệu trên ai. **Reseach chọn hyperbola làm g(ai)**

weight decay: **Research thiết lập nó ở giá trị 5e-4**

learning rate: Bắt đầu với giá trị 0.1 và được giảm xuống 1/10 vào các epoch thứ 10, 18, và 22. Quá trình huấn luyện dừng lại ở 
epoch thứ 25.(do họ train trên dataset lớn nên làm vậy được còn mk chắc chỉ 0.001)

momemtune: 0.9.

![](images/g(x).png)

![](images/mean(g).png)

**Về cách triển khai hàm m(ai)**

![](images/m(ai).png)

**Trong MagFace và các loss dựa trên margin như Argface và CosFace. Vector tâm của mỗi class chính là ma trận trọng số của 1 neutron trong tầng fully connected**

- Đầu tiên label các class sẽ được one hot coding và tầng fully connected layer có số neutron bằng số class cần phân loại.
- Kiến trúc như sau: backbone => embedding (512-D) => Fully connected layer (có n neutron tượng trưng cho n class phục vụ classification) => One_hot code => label
- Để ra được one hot code thì W của các neutron phải thỏa mãn embedding thuộc class nào thì nó sẽ ra gần 1 và các neutron còn lại phải ra gần 0. Và trọng số w của neutron đó chính là vector tâm của class trong hypershpear

In [ ]:
class MagLoss(torch.nn.Module):
    def __init__(self, l_a, u_a, l_margin, u_margin, scale=64.0):
        super(MagLoss, self).__init__()
        self.l_a = l_a
        self.u_a = u_a
        self.scale = scale
        # Được tính từ l_margin để xác định 1 ngưỡng mà tại đó loss bắt đầu thay đổi, cần phải được điều chỉnh để dễ dàng huấn luyện hơn.
        self.cut_off = np.cos(np.pi/2-l_margin)
        # Có thể được sử dụng trong các trường hợp khi một phần của mất mát (loss) cần phải được điều chỉnh để dễ dàng huấn luyện hơn
        self.large_value = 1 << 10

    # Định nghĩa g(ai), trong đó x_norm là giá trị của ai muốn được chuẩn hóa sao cho nó nằm trong [la, ua]
    # la được định nghĩa gián tiếp ở các phần khác
    # g là 1 tensor được tính toán dựa trên magnitude của feature vector x_norm. torch.mean(g) là 1 phép tính giá trị trung bình của tất cả phần tử trong tensor.
    def calc_loss_G(self, x_norm):
        g = 1/(self.u_a**2) * x_norm + 1/(x_norm)
        return torch.mean(g)

    # input là giá trị cosine [cos_theta, cos_theta_m] từ MagLinear
    # target: label của feature
    def forward(self, input, target, x_norm):
        loss_g = self.calc_loss_G(x_norm)

        cos_theta, cos_theta_m = input
        one_hot = torch.zeros_like(cos_theta)
        one_hot.scatter_(1, target.view(-1, 1), 1.0)
        # cos_theta và cos_theta_m được kết hợp lại để tạo thành output
        output = one_hot * cos_theta_m + (1.0 - one_hot) * cos_theta
        # Dùng cross_entropy để tính loss chính
        loss = F.cross_entropy(output, target, reduction='mean')
        # Trả về 3 giá trị: softmax loss (tăng cường tính phân biệt của embedding), loss_g: regularization loss, one_hot: mã one-hot của label.
        return loss.mean(), loss_g, one_hot

In [ ]:
#!/usr/bin/env python
import sys
sys.path.append("..")
from models import iresnet
from collections import OrderedDict
from torch.nn import Parameter
import torch.nn.functional as F
import torch.backends.cudnn as cudnn
import numpy as np
import math
import torch
import torch.nn as nn
import os


def builder(args):
    model = SoftmaxBuilder(args)
    return model


def load_features(args):
    if args.arch == 'iresnet18':
        features = iresnet.iresnet18(
            pretrained=True,
            num_classes=args.embedding_size)
    elif args.arch == 'iresnet34':
        features = iresnet.iresnet34(
            pretrained=True,
            num_classes=args.embedding_size)
    elif args.arch == 'iresnet50':
        features = iresnet.iresnet50(
            pretrained=True,
            num_classes=args.embedding_size)
    elif args.arch == 'iresnet100':
        features = iresnet.iresnet100(
            pretrained=True,
            num_classes=args.embedding_size)
    else:
        raise ValueError()
    return features


class SoftmaxBuilder(nn.Module):
    def __init__(self, args):
        super(SoftmaxBuilder, self).__init__()
        self.features = load_features(args)
        self.fc = MagLinear(args.embedding_size,
                            args.last_fc_size,
                            scale=args.arc_scale)

        self.l_margin = args.l_margin
        self.u_margin = args.u_margin
        self.l_a = args.l_a
        self.u_a = args.u_a

    def _margin(self, x):
        """generate adaptive margin
        """
        margin = (self.u_margin-self.l_margin) / \
            (self.u_a-self.l_a)*(x-self.l_a) + self.l_margin
        return margin

    def forward(self, x, target):
        x = self.features(x)
        logits, x_norm = self.fc(x, self._margin, self.l_a, self.u_a)
        return logits, x_norm


class MagLinear(torch.nn.Module):
    """
    Parallel fc for Mag loss
    """

    def __init__(self, in_features, out_features, scale=64.0, easy_margin=True):
        super(MagLinear, self).__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.weight = Parameter(torch.Tensor(in_features, out_features))
        self.weight.data.uniform_(-1, 1).renorm_(2, 1, 1e-5).mul_(1e5)
        self.scale = scale
        self.easy_margin = easy_margin

    def forward(self, x, m, l_a, u_a):
        """
        Here m is a function which generate adaptive margin
        """
        x_norm = torch.norm(x, dim=1, keepdim=True).clamp(l_a, u_a)
        ada_margin = m(x_norm)
        cos_m, sin_m = torch.cos(ada_margin), torch.sin(ada_margin)

        # norm the weight
        weight_norm = F.normalize(self.weight, dim=0)
        cos_theta = torch.mm(F.normalize(x), weight_norm)
        cos_theta = cos_theta.clamp(-1, 1)
        sin_theta = torch.sqrt(1.0 - torch.pow(cos_theta, 2))
        cos_theta_m = cos_theta * cos_m - sin_theta * sin_m
        if self.easy_margin:
            cos_theta_m = torch.where(cos_theta > 0, cos_theta_m, cos_theta)
        else:
            mm = torch.sin(math.pi - ada_margin) * ada_margin
            threshold = torch.cos(math.pi - ada_margin)
            cos_theta_m = torch.where(
                cos_theta > threshold, cos_theta_m, cos_theta - mm)
        # multiply the scale in advance
        cos_theta_m = self.scale * cos_theta_m
        cos_theta = self.scale * cos_theta

        return [cos_theta, cos_theta_m], x_norm


class MagLoss(torch.nn.Module):
    """
    MagFace Loss.
    """

    def __init__(self, l_a, u_a, l_margin, u_margin, scale=64.0):
        super(MagLoss, self).__init__()
        self.l_a = l_a
        self.u_a = u_a
        self.scale = scale
        self.cut_off = np.cos(np.pi/2-l_margin)
        self.large_value = 1 << 10

    def calc_loss_G(self, x_norm):
        g = 1/(self.u_a**2) * x_norm + 1/(x_norm)
        return torch.mean(g)

    def forward(self, input, target, x_norm):
        loss_g = self.calc_loss_G(x_norm)

        cos_theta, cos_theta_m = input
        one_hot = torch.zeros_like(cos_theta)
        one_hot.scatter_(1, target.view(-1, 1), 1.0)
        output = one_hot * cos_theta_m + (1.0 - one_hot) * cos_theta
        loss = F.cross_entropy(output, target, reduction='mean')
        return loss.mean(), loss_g, one_hot

```sh
#!/usr/bin/env bash

export CUDA_VISIBLE_DEVICES=0,1,2,3,4,5,6,7

la=10
ua=110
lm=0.45
um=0.8
lg=35

# settings
MODEL_ARC=iresnet50
OUTPUT=./test/

mkdir -p ${OUTPUT}/vis/

python -u trainer.py \
    --arch ${MODEL_ARC} \
    --train_list /training/face-group/opensource/ms1m-112/ms1m_train.list \
    --workers 8 \
    --epochs 25 \
    --start-epoch 0 \
    --batch-size 512 \
    --embedding-size 512 \
    --last-fc-size 85742 \
    --arc-scale 64 \
    --learning-rate 0.1 \
    --momentum 0.9 \
    --weight-decay 5e-4 \
    --lr-drop-epoch 10 18 22 \
    --lr-drop-ratio 0.1 \
    --print-freq 100 \
    --pth-save-fold ${OUTPUT} \
    --pth-save-epoch 1 \
    --l_a ${la} \
    --u_a ${ua} \
    --l_margin ${lm} \
    --u_margin ${um} \
    --lambda_g ${lg} \
    --vis_mag 1    2>&1 | tee ${OUTPUT}/output.log   
```

# Tổng quan lại từ đâu

![](images/Tổng%20quan%20MagFace.png)

![](images/Tổng%20quan%20softmaxbuilder.png)

![](images/Tổng%20quan%20MagLinear.png)

![](images/Tổng%20quan%20MagLoss.png)